# 00 — Exploratory Data Analysis

Understand the raw `ScooterIMUData.csv` schema, value distributions, sampling rate,
sensor lag, and any data quality issues before running any algorithms.

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

from scooter_imu.data.loader import load_raw
from scooter_imu.data.preprocessor import preprocess

RAW_PATH = Path("../Data/raw/ScooterIMUData.csv")

In [14]:
raw = load_raw(RAW_PATH)
print(f"Shape: {raw.shape}")
raw.head()

Shape: (9000, 5)


,ts,received_ts,device_uuid,data_item_name,value
0,2024-03-15 10:00:00.000,2024-03-15 10:00:00.007,ather-001,GYR_X_DEG,-4.699921
1,2024-03-15 10:00:00.000,2024-03-15 10:00:00.007,ather-001,GYR_Y_DEG,2.451354
2,2024-03-15 10:00:00.000,2024-03-15 10:00:00.007,ather-001,GYR_Z_DEG,9.405647
3,2024-03-15 10:00:00.020,2024-03-15 10:00:00.044,ather-001,GYR_X_DEG,-9.255176
4,2024-03-15 10:00:00.020,2024-03-15 10:00:00.044,ather-001,GYR_Y_DEG,-3.706539


In [15]:
# Row counts per axis
raw["data_item_name"].value_counts()

data_item_name
GYR_X_DEG    3000
GYR_Y_DEG    3000
GYR_Z_DEG    3000
Name: count, dtype: int64

In [16]:
# Value distribution per axis
for axis in ["GYR_X_DEG", "GYR_Y_DEG", "GYR_Z_DEG"]:
    subset = raw[raw["data_item_name"] == axis]["value"]
    print(f"{axis}: mean={subset.mean():.4f}  std={subset.std():.4f}  "
          f"min={subset.min():.4f}  max={subset.max():.4f}")

GYR_X_DEG: mean=0.4828  std=5.0210  min=-16.5436  max=20.6291
GYR_Y_DEG: mean=0.1886  std=2.9873  min=-12.9673  max=10.5078
GYR_Z_DEG: mean=-0.2858  std=10.2590  min=-36.4841  max=34.5405


In [17]:
df = preprocess(raw)
print(f"Wide shape: {df.shape}")
df.describe()

Wide shape: (3000, 9)


,timestamp,received_ts,gyr_x,gyr_y,gyr_z,sensor_lag_s,gyr_magnitude,dt
count,3000,3000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.00
mean,2024-03-15 10:00:29.990000,2024-03-15 10:00:30.006882,0.482798,0.188598,-0.285784,0.016882,10.409964,0.02
min,2024-03-15 10:00:00,2024-03-15 10:00:00.007000,-16.543560,-12.967344,-36.484128,0.005000,0.335689,0.02
25%,2024-03-15 10:00:14.995000,2024-03-15 10:00:15.007250,-2.842140,-1.827138,-7.010871,0.011000,6.311323,0.02
50%,2024-03-15 10:00:29.990000,2024-03-15 10:00:30.005000,0.492868,0.162467,-0.412142,0.017000,9.377169,0.02
75%,2024-03-15 10:00:44.985000,2024-03-15 10:00:45.009250,3.787425,2.185016,6.476757,0.023000,13.507859,0.02
max,2024-03-15 10:00:59.980000,2024-03-15 10:00:59.989000,20.629120,10.507783,34.540464,0.029000,39.156316,0.02
std,NaN,NaN,5.021001,2.987279,10.258957,0.007231,5.597093,0.00


In [18]:
# Sampling interval distribution
fig = px.histogram(df, x="dt", nbins=80, title="Sampling Interval Distribution (dt)",
                   labels={"dt": "dt (seconds)"})
fig.show()

In [19]:
# Sensor lag
fig = px.histogram(df, x="sensor_lag_s", nbins=80, title="Sensor Lag Distribution",
                   labels={"sensor_lag_s": "lag (seconds)"})
fig.show()

In [20]:
# 3-axis time-series overview
import plotly.graph_objects as go
fig = go.Figure()
for col, color in [("gyr_x", "red"), ("gyr_y", "blue"), ("gyr_z", "green")]:
    fig.add_trace(go.Scatter(x=df["timestamp"], y=df[col], name=col, line=dict(color=color, width=1)))
fig.update_layout(title="All Gyroscope Axes Over Time", xaxis_title="Time", yaxis_title="deg/s")
fig.show()